"""onyxia@vscode-python-gpu-336003-0:~/work$ cd DLforTimeSeries/
onyxia@vscode-python-gpu-336003-0:~/work/DLforTimeSeries$ cd moirai-classification/
onyxia@vscode-python-gpu-336003-0:~/work/DLforTimeSeries/moirai-classification$ uv sync
Resolved 146 packages in 1ms
Audited 141 packages in 1ms
onyxia@vscode-python-gpu-336003-0:~/work/DLforTimeSeries/moirai-classification$ uv run main.py"""
#puis chemin vers 3.11

In [ ]:

import numpy as np

from tslearn.datasets import UCR_UEA_datasets
import torch
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.linear_model import LogisticRegression
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from uni2ts.model.moirai import MoiraiModule
import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import RidgeClassifierCV

from encoder import MoiraiEncoder

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data

Import

In [2]:
device = "cuda"
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

n_classes = len(set(y_train))
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

Prepare data

In [3]:
def preprocess_data(
    data: np.ndarray,
    *,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
):
    """
    data: np.ndarray of shape (N, T, V) = (n_individual, time, variate)
    Assumes NO missing values and NO padding in the raw data.
    """

    if not isinstance(data, np.ndarray):
        raise TypeError("data must be a numpy.ndarray")
    if data.ndim != 3:
        raise ValueError(f"Expected shape (N,T,V), got {data.shape}")

    N, T, V = data.shape

    # (N,T,V)
    past_target = torch.as_tensor(data, dtype=dtype, device=device)

    # observed mask: (N,T,V) all True no missing values
    past_observed_target = torch.ones((N, T, V), dtype=torch.bool, device=device)

    # padding mask: (N,T) all if no padding
    past_is_pad = torch.zeros((N, T), dtype=torch.bool, device=device)

    return past_target, past_observed_target, past_is_pad

In [4]:
X_train_target_tensor, X_train_is_target_observed, X_train_is_target_padded = (
    preprocess_data(X_train, device=device)
)

X_test_target_tensor, X_test_is_target_observed, X_test_is_target_padded = (
    preprocess_data(X_test, device=device)
)


print(X_train_target_tensor.shape)
print(X_train_is_target_observed.shape)
print(X_train_is_target_padded.shape)

torch.Size([2459, 36, 6])
torch.Size([2459, 36, 6])
torch.Size([2459, 36])


Creat Z_train and Z_test for each patch size

In [5]:
# Paramètres à tester
patch_sizes = [8, 16, 32, 64]

SIZE = "small"
device = "cuda"

Z_train_patch = {}
Z_test_patch = {}
for patch in patch_sizes:
    
    # 1. Définir les paramètres et l'encodeur pour la taille de patch actuelle
    MODEL_PARAMS = {
        "patch_size": patch,
        "num_samples": 100,
        "target_dim": 1,
        "feat_dynamic_real_dim": 0,
        "past_feat_dynamic_real_dim": 0,
    }
    
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=0,
        context_length=36,
        **MODEL_PARAMS,
    )
    encoder.eval()
    encoder.to(device)
    
    # 2. Extraire les embeddings (Z_train et Z_test)
    with torch.no_grad():
        Z_train = encoder(
            past_target=X_train_target_tensor,
            past_observed_target=X_train_is_target_observed,
            past_is_pad=X_train_is_target_padded,
        )
        Z_test = encoder(
            past_target=X_test_target_tensor,
            past_observed_target=X_test_is_target_observed,
            past_is_pad=X_test_is_target_padded,
        )
        
    Z_train_patch[patch] = Z_train
    Z_test_patch[patch] = Z_test

In [6]:

print(f"{'Patch Size':<12} | {'Train Shape':<25} | {'Test Shape':<25}")
for patch in patch_sizes:
    train_tensor = Z_train_patch[patch]
    test_tensor = Z_test_patch[patch]
    print(f"{patch:<12} | {str(list(train_tensor.shape)):<25} | {str(list(test_tensor.shape)):<25}")

Patch Size   | Train Shape               | Test Shape               
8            | [2459, 30, 384]           | [2466, 30, 384]          
16           | [2459, 18, 384]           | [2466, 18, 384]          
32           | [2459, 12, 384]           | [2466, 12, 384]          
64           | [2459, 6, 384]            | [2466, 6, 384]           


In [7]:
encoder.module

MoiraiModule(
  (mask_encoding): Embedding(1, 384)
  (scaler): PackedStdScaler()
  (in_proj): MultiInSizeLinear(in_features_ls=[8, 16, 32, 64, 128], out_features=384, bias=True, dtype=torch.float32)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): GroupedQueryAttention(
          (var_attn_bias): BinaryAttentionBias(
            (emb): Embedding(2, 6)
          )
          (time_qk_proj): QueryKeyProjection(
            (query_proj): RotaryProjection()
            (key_proj): RotaryProjection()
          )
          (q_proj): Linear(in_features=384, out_features=384, bias=False)
          (k_proj): Linear(in_features=384, out_features=384, bias=False)
          (v_proj): Linear(in_features=384, out_features=384, bias=False)
          (q_norm): RMSNorm(normalized_shape=(64,), eps=1e-05, weight=True)
          (k_norm): RMSNorm(normalized_shape=(64,), eps=1e-05, weight=True)
          (out_proj): Linear(in_features=

# Basic pooling with logistic regression

## Single patch pooling

We started by applying simple pooling techniques (max, min, or mean) and comparing their performance across various patch sizes. Specifically, we evaluated whether to apply this pooling globally across all patches, locally within a single series, or across corresponding patches from all series.

### Training

Define one function for every type of pooling

In [11]:
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=device)

num_classes = len(torch.unique(y_train_tensor))
NUM_VARS = 6

def apply_pooling_pt(Z_tensor, method, num_vars=NUM_VARS):
    N, S, F = Z_tensor.shape
    P = S // num_vars # Calcul automatique du nombre de patches par variable
    
    # On reshape le tenseur pour séparer les Variables et les Patches
    # Forme résultante : (Batch, Variables, Patches, Features)
    Z_reshaped = Z_tensor.view(N, num_vars, P, F)
    
    # Basique et Global
    if method == "flatten":
        return Z_tensor.reshape(N, -1)
        
    elif method == "global_mean":
        return Z_tensor.mean(dim=1)
        
    elif method == "global_max":
        return Z_tensor.max(dim=1).values
        
    elif method == "global_min":
        return Z_tensor.min(dim=1).values
    
    elif method == "global_mean_max_min":
        return torch.cat([
            Z_tensor.mean(dim=1),
            Z_tensor.max(dim=1).values,
            Z_tensor.min(dim=1).values
        ], dim=1)

    # Pooling sur les Patches (on garde les variables distinctes) ---
    # Réduction sur la dimension 2 (Patches). Résultat : (N, num_vars, F), puis on aplatit
    elif method == "mean_over_patches":
        return Z_reshaped.mean(dim=2).reshape(N, -1)
        
    elif method == "max_over_patches":
        return Z_reshaped.max(dim=2).values.reshape(N, -1)
        
    elif method == "min_over_patches":
        return Z_reshaped.min(dim=2).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_patches":
        p_mean = Z_reshaped.mean(dim=2).reshape(N, -1)
        p_max  = Z_reshaped.max(dim=2).values.reshape(N, -1)
        p_min  = Z_reshaped.min(dim=2).values.reshape(N, -1)
        return torch.cat([p_mean, p_max, p_min], dim=1)

    # Pooling sur les Variables (on synchronise les patches entre variables) ---
    # Réduction sur la dimension 1 (Variables). Résultat : (N, P, F), puis on aplatit
    elif method == "mean_over_variables":
        return Z_reshaped.mean(dim=1).reshape(N, -1)
        
    elif method == "max_over_variables":
        return Z_reshaped.max(dim=1).values.reshape(N, -1)
        
    elif method == "min_over_variables":
        return Z_reshaped.min(dim=1).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_variables":
        v_mean = Z_reshaped.mean(dim=1).reshape(N, -1)
        v_max  = Z_reshaped.max(dim=1).values.reshape(N, -1)
        v_min  = Z_reshaped.min(dim=1).values.reshape(N, -1)
        return torch.cat([v_mean, v_max, v_min], dim=1)

    else:
        raise ValueError(f"Method {method} unknow")

Our machine have GPU so we do a Logistic Regression  with pytorch to be faster

In [10]:
class LogisticRegressionPT(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(LogisticRegressionPT, self).__init__()
        self.linear = nn.Linear(input_dim, num_classes)
        
    def forward(self, x):
        return self.linear(x)

We creat a dataframe to stock the results and we fill it

In [10]:
pooling_methods = [
    "flatten", 
    "global_mean", "global_max", "global_min", "global_mean_max_min",
    "mean_over_patches", "max_over_patches", "min_over_patches", "mean_max_min_over_patches",
    "mean_over_variables", "max_over_variables", "min_over_variables", "mean_max_min_over_variables"
]

alphas_to_test = [0.1,1,10,30,100,300,1000,10000]

df_results = pd.DataFrame(index=pooling_methods, columns=patch_sizes)
df_results.index.name = "Pooling \\ Patch"

for patch in patch_sizes:
    Z_train = Z_train_patch[patch].to(device)
    Z_test = Z_test_patch[patch].to(device)
    
    for pool in pooling_methods:
        # Appliquer le pooling avec la nouvelle logique
        X_train_pool_tensor = apply_pooling_pt(Z_train, pool)
        X_test_pool_tensor = apply_pooling_pt(Z_test, pool)


        # Convertir en NumPy pour Scikit-Learn (on repasse sur CPU)
        X_train_pool = X_train_pool_tensor.cpu().numpy()
        X_test_pool = X_test_pool_tensor.cpu().numpy()

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_pool)
        X_test_scaled = scaler.transform(X_test_pool)
        
        # Entraînement avec Scikit-Learn
        # max_iter=1000 pour éviter les warnings de non-convergence
        clf = RidgeClassifierCV(alphas=alphas_to_test, cv = 3)#LogisticRegression(random_state=42, C = 1, max_iter=200)
        clf.fit(X_train_pool, y_train) # Note: on utilise directement y_train (numpy), pas y_train_tensor
        # Évaluation
        y_pred = clf.predict(X_test_pool)
        acc = accuracy_score(y_test, y_pred)
        print(pool)
        print(clf.alpha_)
            
        
        """
        
        input_dim = X_train_pool.shape[1]
        
        # Initialiser le modèle
        model = LogisticRegressionPT(input_dim, num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.01)
        epochs = 200
        model.train()
        for epoch in range(epochs):
            optimizer.zero_grad()
            outputs = model(X_train_pool)
            loss = criterion(outputs, y_train_tensor)
            loss.backward()
            optimizer.step()
            
        # Évaluation
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test_pool)
            _, predicted = torch.max(test_outputs.data, 1)
            
            correct = (predicted == y_test_tensor).sum().item()
            total = y_test_tensor.size(0)
            acc = correct / total

        """    
        # Remplissage discret du DataFrame
        df_results.loc[pool, patch] = round(acc, 3)

flatten
1000.0
global_mean
1.0
global_max
1.0
global_min
10.0
global_mean_max_min
100.0


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=5.8896e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)


mean_over_patches
30.0


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.72297e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.43347e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.27428e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)


max_over_patches
300.0
min_over_patches
100.0
mean_max_min_over_patches
300.0
mean_over_variables
30.0


...

max_over_patches
100.0


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.0732e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.1347e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.14779e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)


min_over_patches
100.0


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.61689e-09): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.92965e-09): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.63316e-09): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditio

mean_max_min_over_patches
300.0
mean_over_variables
10.0
max_over_variables
1.0
min_over_variables
10.0
mean_max_min_over_variables
10.0


### Results

In [11]:
# Affichage propre du DataFrame
df_results = df_results.astype(float)
display(df_results)  # Remplace par print(df_results) si display n'est pas supporté

# Recherche de la meilleure combinaison
best_acc = df_results.max().max()
best_pool, best_patch = df_results.stack().idxmax()


print(f"MEILLEURE COMBINAISON : Patch = {best_patch} | Pooling = '{best_pool}'")
print(f"Précision maximale : {best_acc:.3f}")

,8,16,32,64
Pooling \ Patch,,,,
flatten,0.579,0.587,0.595,0.604
global_mean,0.582,0.586,0.571,0.571
global_max,0.533,0.528,0.530,0.562
global_min,0.526,0.513,0.525,0.564
global_mean_max_min,0.556,0.563,0.555,0.566
mean_over_patches,0.599,0.581,0.584,0.604
max_over_patches,0.575,0.578,0.572,0.604
min_over_patches,0.579,0.582,0.571,0.604
mean_max_min_over_patches,0.599,0.588,0.576,0.604


MEILLEURE COMBINAISON : Patch = 64 | Pooling = 'flatten'
Précision maximale : 0.604


We observe that smaller patch sizes of 8 yield better results by capturing more granular information, and that minimizing cross-variable aggregation by pooling over patches rather than series produces the highest accuracy.

## Multi Patch pooling

### Training

In [ ]:
import numpy as np

# On crée un DataFrame avec une seule colonne pour les résultats combinés
df_multiscale_results = pd.DataFrame(index=pooling_methods, columns=["All_Patches_Combined"])
df_multiscale_results.index.name = "Pooling Method"

for pool in pooling_methods:
    X_train_multi = []
    X_test_multi = []
    
    # On boucle sur chaque taille de patch pour extraire et pooler les features
    for patch in patch_sizes:
        Z_train = Z_train_patch[patch].to(device)
        Z_test = Z_test_patch[patch].to(device)
        
        # Appliquer le pooling et passer en NumPy
        X_train_pool = apply_pooling_pt(Z_train, pool).cpu().numpy()
        X_test_pool = apply_pooling_pt(Z_test, pool).cpu().numpy()
        
        X_train_multi.append(X_train_pool)
        X_test_multi.append(X_test_pool)
        
    # ETAPE CLÉ : Concaténer toutes les features sur l'axe des variables (axis=1)
    # Si chaque patch donne un vecteur de taille F, on aura un vecteur de taille F * 4
    X_train_combined = np.concatenate(X_train_multi, axis=1)
    X_test_combined = np.concatenate(X_test_multi, axis=1)

    print(pool)
    print(clf.alpha_)
    
    # Entraînement avec Scikit-Learn sur les features combinées
    clf = RidgeClassifierCV(alphas=alphas_to_test, cv = 3)#LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train_combined, y_train)
    
    # Évaluation
    y_pred = clf.predict(X_test_combined)
    acc = accuracy_score(y_test, y_pred)
        
    # Remplissage du DataFrame
    df_multiscale_results.loc[pool, "All_Patches_Combined"] = round(acc, 3)

flatten
10.0
global_mean
1000.0
global_max
10.0
global_min
100.0
global_mean_max_min
100.0
mean_over_patches
1000.0
max_over_patches
300.0
min_over_patches
1000.0
mean_max_min_over_patches
1000.0


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.46086e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.91297e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.58398e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)


mean_over_variables
1000.0
max_over_variables
100.0
min_over_variables
100.0
mean_max_min_over_variables
300.0


### Results

In [13]:
display(df_multiscale_results.astype(float))

best_acc_multi = df_multiscale_results.max().max()
best_pool_multi = df_multiscale_results.idxmax()[0]
print(f"MEILLEURE COMBINAISON MULTI-ÉCHELLE : Pooling = '{best_pool_multi}'")
print(f"Précision maximale : {best_acc_multi:.2f}")

,All_Patches_Combined
Pooling Method,
flatten,0.61
global_mean,0.61
global_max,0.58
global_min,0.58
global_mean_max_min,0.59
mean_over_patches,0.63
max_over_patches,0.62
min_over_patches,0.62
mean_max_min_over_patches,0.63


MEILLEURE COMBINAISON MULTI-ÉCHELLE : Pooling = 'mean_over_patches'
Précision maximale : 0.63


/tmp/ipykernel_2512/1661945645.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  best_pool_multi = df_multiscale_results.idxmax()[0]


# Advanced Pooling

## Generalized Mean (GeM) Pooling (Dont work yet)

Unlike standard methods (like mean or max), Generalized Mean (GeM) Pooling introduces a trainable parameter $p$ that the neural network learns during backpropagation. 
The formula is: 
$$f = \left(\frac{1}{N} \sum |x_i|^p\right)^\frac{1}{p}$$

To find the best approach, we will test GeM pooling across different dimensions dynamically during training:
- **Global**: Pooling over the entire sequence at once.
- **Over Patches**: Keeping variables separate and pooling their respective patches.
- **Over Variables**: Synchronizing patches and pooling across the different series.

For more details, you can refer to the original paper: [Fine-tuning CNN Image Retrieval with No Human Annotation (Radenović et al.)](https://arxiv.org/abs/1711.02512).

In [14]:
class GeMPooling(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super(GeMPooling, self).__init__()
        # The parameter p is learnable by the optimizer
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x, dim):
        # Clamp values to avoid NaN errors with fractional powers
        x = x.clamp(min=self.eps)
        return x.pow(self.p).mean(dim=dim).pow(1.0 / self.p)


class DynamicGeMClassifierPT(nn.Module):
    def __init__(self, num_vars, num_classes):
        super(DynamicGeMClassifierPT, self).__init__()
        self.num_vars = num_vars
        self.gem = GeMPooling(p=3.0)
        
        # LazyLinear automatically infers the input size during the first forward pass.
        # This allows us to dynamically change the pooling dimension without crashing!
        self.linear = nn.LazyLinear(num_classes)
        
    def forward(self, z, pool_dim):
        N, S, F = z.shape
        
        if pool_dim == "global":
            # Pool over the entire sequence dimension (dim=1)
            pooled = self.gem(z, dim=1)
            
        else:
            P = S // self.num_vars
            # Reshape: (Batch, Variables, Patches, Features)
            z_reshaped = z.view(N, self.num_vars, P, F)
            
            if pool_dim == "patches":
                # Pool over the Patches dimension (dim=2)
                pooled = self.gem(z_reshaped, dim=2)
            elif pool_dim == "variables":
                # Pool over the Variables dimension (dim=1)
                pooled = self.gem(z_reshaped, dim=1)
            else:
                raise ValueError("pool_dim must be 'global', 'patches', or 'variables'")
        
        # Flatten for the linear layer
        pooled_flat = pooled.view(N, -1)
        return self.linear(pooled_flat)

### Dynamic Training and Evaluation

We will now train the GeM model by iterating over both the `patch_sizes` and the `pooling_dimensions`. We will also monitor what value of $p$ the network naturally converged to for each configuration.

In [15]:
pooling_dimensions = ["global", "patches", "variables"]
df_gem_results = pd.DataFrame(index=pooling_dimensions, columns=patch_sizes)
df_gem_results.index.name = "GeM Pooling Dimension"

for patch in patch_sizes:
    Z_train = Z_train_patch[patch].to(device)
    Z_test = Z_test_patch[patch].to(device)
    
    for pool_dim in pooling_dimensions:
        # Instantiate the model
        model = DynamicGeMClassifierPT(num_vars=NUM_VARS, num_classes=num_classes).to(device)
        
        # IMPORTANT: We must perform a "dummy" forward pass with a tiny batch
        # to initialize the LazyLinear weights before creating the optimizer.
        _ = model(Z_train[:2], pool_dim=pool_dim)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.01)
        
        epochs = 500
        model.train()
        for epoch in range(epochs):
            optimizer.zero_grad()
            # We explicitly tell the model which dimension to pool over
            outputs = model(Z_train, pool_dim=pool_dim) 
            loss = criterion(outputs, y_train_tensor)
            loss.backward()
            optimizer.step()
            
        # Evaluation
        model.eval()
        with torch.no_grad():
            test_outputs = model(Z_test, pool_dim=pool_dim)
            _, predicted = torch.max(test_outputs.data, 1)
            
            correct = (predicted == y_test_tensor).sum().item()
            total = y_test_tensor.size(0)
            acc = correct / total
            
            # Retrieve the learned p value
            learned_p = model.gem.p.item()
            
        df_gem_results.loc[pool_dim, patch] = round(acc, 2)
        print(f"Patch: {patch:<4} | Dim: {pool_dim:<10} | Test Acc: {acc:.2f} | Learned p: {learned_p:.2f}")


display(df_gem_results.astype(float))

Patch: 8    | Dim: global     | Test Acc: 0.56 | Learned p: 6.49
Patch: 8    | Dim: patches    | Test Acc: 0.60 | Learned p: 5.58
Patch: 8    | Dim: variables  | Test Acc: 0.53 | Learned p: 5.71
Patch: 16   | Dim: global     | Test Acc: 0.54 | Learned p: 6.25
Patch: 16   | Dim: patches    | Test Acc: 0.57 | Learned p: 5.29
Patch: 16   | Dim: variables  | Test Acc: 0.51 | Learned p: 6.30
Patch: 32   | Dim: global     | Test Acc: 0.51 | Learned p: 6.47
Patch: 32   | Dim: patches    | Test Acc: 0.55 | Learned p: 5.50
Patch: 32   | Dim: variables  | Test Acc: 0.51 | Learned p: 6.81
Patch: 64   | Dim: global     | Test Acc: 0.54 | Learned p: 6.67
Patch: 64   | Dim: patches    | Test Acc: 0.58 | Learned p: 3.08
Patch: 64   | Dim: variables  | Test Acc: 0.54 | Learned p: 6.76


,8,16,32,64
GeM Pooling Dimension,,,,
global,0.56,0.54,0.51,0.54
patches,0.60,0.57,0.55,0.58
variables,0.53,0.51,0.51,0.54


## Attention Pooling

In [ ]:
class DynamicAttentionClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, patches_counts, in_features=384, mode="global"):
        """
        patches_counts : liste du nombre de patchs par échelle (ex: [30, 18, 12, 6])
        mode : "global" (1), "per_scale" (4), "per_variable" (6), ou "per_both" (24)
        """
        super().__init__()
        self.num_vars = num_vars
        self.patches_counts = patches_counts
        self.num_scales = len(patches_counts)
        self.mode = mode
        
        if mode == "global":
            self.context = nn.Parameter(torch.randn(in_features) * 0.1)
            
        elif mode == "per_scale":
            self.context = nn.Parameter(torch.randn(self.num_scales, in_features) * 0.1)
            
        elif mode == "per_variable":
            self.context = nn.Parameter(torch.randn(self.num_vars, in_features) * 0.1)
            
        elif mode == "per_both":
            self.context = nn.Parameter(torch.randn(self.num_scales, self.num_vars, in_features) * 0.1)
            
            
        self.dropout = nn.Dropout(0.1)
        self.linear = nn.LazyLinear(num_classes)
        
    def forward(self, x):
        B = x.shape[0]
        F = x.shape[-1]
        
        split_sizes = [p * self.num_vars for p in self.patches_counts]
        x_splits = torch.split(x, split_sizes, dim=1)
        
        reshaped_splits = []
        for i, p in enumerate(self.patches_counts):
            reshaped_splits.append(x_splits[i].view(B, self.num_vars, p, F))
            
        if self.mode == "global":
            scores = torch.matmul(x, self.context)
            weights = torch.softmax(scores, dim=1).unsqueeze(-1)
            pooled = (weights * x).sum(dim=1)
            
        elif self.mode == "per_scale":
            pooled_list = []
            for i, xi in enumerate(reshaped_splits):
                xi_flat = xi.view(B, -1, F)
                scores = torch.matmul(xi_flat, self.context[i])
                weights = torch.softmax(scores, dim=1).unsqueeze(-1)
                pooled_list.append((weights * xi_flat).sum(dim=1))
            
            pooled = torch.cat(pooled_list, dim=1)
            
        elif self.mode == "per_variable":
            x_vars = torch.cat(reshaped_splits, dim=2)
            
            scores = (x_vars * self.context.view(1, self.num_vars, 1, F)).sum(dim=-1)
            weights = torch.softmax(scores, dim=2).unsqueeze(-1)
            pooled_vars = (weights * x_vars).sum(dim=2)
            pooled = pooled_vars.view(B, -1)
            
        elif self.mode == "per_both":
            pooled_list = []
            for i, xi in enumerate(reshaped_splits):
                scores = (xi * self.context[i].view(1, self.num_vars, 1, F)).sum(dim=-1)
                weights = torch.softmax(scores, dim=2).unsqueeze(-1)
                pooled_vars = (weights * xi).sum(dim=2)
                pooled_list.append(pooled_vars)
                
            pooled_all = torch.stack(pooled_list, dim=1)
            pooled = pooled_all.view(B, -1)
            
        return self.linear(self.dropout(pooled))

In [18]:
patch_sizes = [8, 16, 32, 64]
NUM_VARS = 6

patches_counts = [Z_train_patch[p].shape[1] // NUM_VARS for p in patch_sizes]
print(f"Nomber of patch for each scale : {patches_counts}")

Z_train_concat = torch.cat([Z_train_patch[p].to(device) for p in patch_sizes], dim=1)
Z_test_concat  = torch.cat([Z_test_patch[p].to(device) for p in patch_sizes], dim=1)

print(f"Shape Z_train: {Z_train_concat.shape}")

Nomber of patch for each scale : [5, 3, 2, 1]
Shape Z_train: torch.Size([2459, 66, 384])


In [19]:
BATCH_SIZE = 64
EPOCHS = 1000
LEARNING_RATE = 0.0001
EVAL_EVERY_N_EPOCHS = 10
PRINT_LOSS_EVERY = 5 
attention_modes = ['global', 'per_scale', 'per_variable', 'per_both']
df_results = pd.DataFrame(index=attention_modes, columns=["Test Accuracy"])
df_results.index.name = "Attention mode"

train_dataset = TensorDataset(Z_train_concat, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

mode = "global"

model = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(device)

_ = model(Z_train_concat[:2]) # Dummy pass for LazyLinear

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_z, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_z)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_z.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    

    if (epoch + 1) == 1 or (epoch + 1) % PRINT_LOSS_EVERY == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train Loss : {epoch_loss:.4f}", end="")
        
        if (epoch + 1) % EVAL_EVERY_N_EPOCHS == 0:
            model.eval()
            with torch.no_grad():
                test_outputs = model(Z_test_concat)
                _, predicted = torch.max(test_outputs.data, 1)
                acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
            print(f" | Test Acc : {acc:.4f}")
        else:
            print() 
            
model.eval()
with torch.no_grad():
    test_outputs = model(Z_test_concat)
    _, predicted = torch.max(test_outputs.data, 1)
    final_acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    
df_results.loc[mode, "Test Accuracy"] = round(final_acc, 3)

Epoch [  1/1000] | Train Loss : 2.5732
Epoch [  5/1000] | Train Loss : 2.2269
Epoch [ 10/1000] | Train Loss : 2.0220 | Test Acc : 0.3277
Epoch [ 15/1000] | Train Loss : 1.9158
Epoch [ 20/1000] | Train Loss : 1.8305 | Test Acc : 0.4205
Epoch [ 25/1000] | Train Loss : 1.7574
Epoch [ 30/1000] | Train Loss : 1.7009 | Test Acc : 0.4615
Epoch [ 35/1000] | Train Loss : 1.6577
Epoch [ 40/1000] | Train Loss : 1.6227 | Test Acc : 0.4911
Epoch [ 45/1000] | Train Loss : 1.5930
Epoch [ 50/1000] | Train Loss : 1.5653 | Test Acc : 0.5085
Epoch [ 55/1000] | Train Loss : 1.5364
Epoch [ 60/1000] | Train Loss : 1.5186 | Test Acc : 0.5207
Epoch [ 65/1000] | Train Loss : 1.5015
Epoch [ 70/1000] | Train Loss : 1.4811 | Test Acc : 0.5304
Epoch [ 75/1000] | Train Loss : 1.4629
Epoch [ 80/1000] | Train Loss : 1.4538 | Test Acc : 0.5381
Epoch [ 85/1000] | Train Loss : 1.4439
Epoch [ 90/1000] | Train Loss : 1.4248 | Test Acc : 0.5430
Epoch [ 95/1000] | Train Loss : 1.4187
Epoch [100/1000] | Train Loss : 1.4060 |

In [20]:
mode = "per_scale"

model = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(device)

_ = model(Z_train_concat[:2]) # Dummy pass for LazyLinear

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_z, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_z)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_z.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    

    if (epoch + 1) == 1 or (epoch + 1) % PRINT_LOSS_EVERY == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train Loss : {epoch_loss:.4f}", end="")
        
        if (epoch + 1) % EVAL_EVERY_N_EPOCHS == 0:
            model.eval()
            with torch.no_grad():
                test_outputs = model(Z_test_concat)
                _, predicted = torch.max(test_outputs.data, 1)
                acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
            print(f" | Test Acc : {acc:.4f}")
        else:
            print() 
            
model.eval()
with torch.no_grad():
    test_outputs = model(Z_test_concat)
    _, predicted = torch.max(test_outputs.data, 1)
    final_acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    
df_results.loc[mode, "Test Accuracy"] = round(final_acc, 3)

Epoch [  1/1000] | Train Loss : 2.4546
Epoch [  5/1000] | Train Loss : 1.8087
Epoch [ 10/1000] | Train Loss : 1.6112 | Test Acc : 0.5024
Epoch [ 15/1000] | Train Loss : 1.4951
Epoch [ 20/1000] | Train Loss : 1.4152 | Test Acc : 0.5539
Epoch [ 25/1000] | Train Loss : 1.3630
Epoch [ 30/1000] | Train Loss : 1.3198 | Test Acc : 0.5746
Epoch [ 35/1000] | Train Loss : 1.2825
Epoch [ 40/1000] | Train Loss : 1.2541 | Test Acc : 0.5835
Epoch [ 45/1000] | Train Loss : 1.2284
Epoch [ 50/1000] | Train Loss : 1.2058 | Test Acc : 0.5908
Epoch [ 55/1000] | Train Loss : 1.1881
Epoch [ 60/1000] | Train Loss : 1.1654 | Test Acc : 0.5929
Epoch [ 65/1000] | Train Loss : 1.1436
Epoch [ 70/1000] | Train Loss : 1.1357 | Test Acc : 0.5957
Epoch [ 75/1000] | Train Loss : 1.1182
Epoch [ 80/1000] | Train Loss : 1.1063 | Test Acc : 0.5985
Epoch [ 85/1000] | Train Loss : 1.0902
Epoch [ 90/1000] | Train Loss : 1.0761 | Test Acc : 0.5969
Epoch [ 95/1000] | Train Loss : 1.0624
Epoch [100/1000] | Train Loss : 1.0518 |

In [21]:
mode = "per_variable"

model = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(device)

_ = model(Z_train_concat[:2]) # Dummy pass for LazyLinear

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_z, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_z)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_z.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    

    if (epoch + 1) == 1 or (epoch + 1) % PRINT_LOSS_EVERY == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train Loss : {epoch_loss:.4f}", end="")
        
        if (epoch + 1) % EVAL_EVERY_N_EPOCHS == 0:
            model.eval()
            with torch.no_grad():
                test_outputs = model(Z_test_concat)
                _, predicted = torch.max(test_outputs.data, 1)
                acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
            print(f" | Test Acc : {acc:.4f}")
        else:
            print() 
            
model.eval()
with torch.no_grad():
    test_outputs = model(Z_test_concat)
    _, predicted = torch.max(test_outputs.data, 1)
    final_acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    
df_results.loc[mode, "Test Accuracy"] = round(final_acc, 3)

Epoch [  1/1000] | Train Loss : 2.4016
Epoch [  5/1000] | Train Loss : 1.8149
Epoch [ 10/1000] | Train Loss : 1.5742 | Test Acc : 0.5134
Epoch [ 15/1000] | Train Loss : 1.4266
Epoch [ 20/1000] | Train Loss : 1.3243 | Test Acc : 0.5608
Epoch [ 25/1000] | Train Loss : 1.2464
Epoch [ 30/1000] | Train Loss : 1.1873 | Test Acc : 0.5864
Epoch [ 35/1000] | Train Loss : 1.1397
Epoch [ 40/1000] | Train Loss : 1.1004 | Test Acc : 0.5985
Epoch [ 45/1000] | Train Loss : 1.0611
Epoch [ 50/1000] | Train Loss : 1.0281 | Test Acc : 0.6099
Epoch [ 55/1000] | Train Loss : 0.9992
Epoch [ 60/1000] | Train Loss : 0.9723 | Test Acc : 0.6107
Epoch [ 65/1000] | Train Loss : 0.9433
Epoch [ 70/1000] | Train Loss : 0.9211 | Test Acc : 0.6119
Epoch [ 75/1000] | Train Loss : 0.9024
Epoch [ 80/1000] | Train Loss : 0.8755 | Test Acc : 0.6131
Epoch [ 85/1000] | Train Loss : 0.8564
Epoch [ 90/1000] | Train Loss : 0.8326 | Test Acc : 0.6156
Epoch [ 95/1000] | Train Loss : 0.8220
Epoch [100/1000] | Train Loss : 0.8044 |

In [22]:
mode = "per_both"

model = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(device)

_ = model(Z_train_concat[:2]) # Dummy pass for LazyLinear

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_z, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_z)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_z.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    

    if (epoch + 1) == 1 or (epoch + 1) % PRINT_LOSS_EVERY == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train Loss : {epoch_loss:.4f}", end="")
        
        if (epoch + 1) % EVAL_EVERY_N_EPOCHS == 0:
            model.eval()
            with torch.no_grad():
                test_outputs = model(Z_test_concat)
                _, predicted = torch.max(test_outputs.data, 1)
                acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
            print(f" | Test Acc : {acc:.4f}")
        else:
            print() 
            
model.eval()
with torch.no_grad():
    test_outputs = model(Z_test_concat)
    _, predicted = torch.max(test_outputs.data, 1)
    final_acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    
df_results.loc[mode, "Test Accuracy"] = round(final_acc, 3)

Epoch [  1/1000] | Train Loss : 2.0565
Epoch [  5/1000] | Train Loss : 1.2490
Epoch [ 10/1000] | Train Loss : 1.0207 | Test Acc : 0.6225
Epoch [ 15/1000] | Train Loss : 0.8901
Epoch [ 20/1000] | Train Loss : 0.7820 | Test Acc : 0.6403
Epoch [ 25/1000] | Train Loss : 0.7095
Epoch [ 30/1000] | Train Loss : 0.6394 | Test Acc : 0.6464
Epoch [ 35/1000] | Train Loss : 0.5824
Epoch [ 40/1000] | Train Loss : 0.5296 | Test Acc : 0.6387
Epoch [ 45/1000] | Train Loss : 0.4894
Epoch [ 50/1000] | Train Loss : 0.4488 | Test Acc : 0.6346
Epoch [ 55/1000] | Train Loss : 0.4175
Epoch [ 60/1000] | Train Loss : 0.3866 | Test Acc : 0.6387
Epoch [ 65/1000] | Train Loss : 0.3612
Epoch [ 70/1000] | Train Loss : 0.3359 | Test Acc : 0.6322
Epoch [ 75/1000] | Train Loss : 0.3099
Epoch [ 80/1000] | Train Loss : 0.2878 | Test Acc : 0.6302
Epoch [ 85/1000] | Train Loss : 0.2684
Epoch [ 90/1000] | Train Loss : 0.2525 | Test Acc : 0.6318
Epoch [ 95/1000] | Train Loss : 0.2386
Epoch [100/1000] | Train Loss : 0.2252 |

In [23]:
display(df_results)

,Test Accuracy
Attention mode,
global,0.592
per_scale,0.555
per_variable,0.558
per_both,0.599


TODO :
- target_dim?
- result diff de maxime car moi moi pytorch pour reglog